# Homework 4: Speaker Verification

Send your work by email (or a link to the disk). Don't forget to attach model artifacts.

You can receive for this homework:

- **Practice** - 20 points + 6 bonus
- **Theory** - 4 bonus points

The contribution $G$ of current homework to the final grade for the course is calculated as $G = \frac{N}{10}$, where $N$ - earned points.

So, if you do only the practice completely (20 points), you will get $G = 2$.

# Theory (4 bonus points)

## 1. Estimating Mutual Information with InfoNCE (3 bonus points)

**Small Recap:**

1. Explicit formula of Mutial Information between the original signal $x$ and and the encoding $c$:

$$
I (x | c) = \sum_{x,c} p(x, c) \log \frac{p(x|c)}{p(x)}
$$

2. [InfoNCE Loss](https://arxiv.org/pdf/1807.03748):

$$
L_N = - \mathbf{E}_x \left[\log \frac{f_k \left(x_{t+k}, c_t\right)}{\sum_{x_j \in X}{f_k (x_j, c_t)}} \right],
$$
where $f_k(x_{t+k}, c_t) \sim \frac{p(x_{t + k}|c_t)}{p(x_{t+k})}$


**Task:** Provide some evidence with explanations that

$$I(x_{t + k}, c_t) \ge \log(N) - L_N$$

How the results could be interpreted in practice?

## 2. Triplet Loss vs Contrastive Loss (1 bonus point)

**Small Recap:**


1. **Triplet Loss**. For a given anchor $A$, a positive sample $P$ (same class as $A$), and a negative sample $N$ (different class from $A$):

$$
L_{triplet}(A, P, N) = \max\left(0,\ \|A - P\|^2 - \|A - N\|^2 + \alpha\right)
$$

2. **Contrastive Loss**. For a pair of points $x_1, x_2$ with label $y \in \{0,1\} $, ($y=0$ means "different classes" and $y=1$ means "same classes"):

$$
L_{contrastive}(x_1, x_2, y) = y \cdot \frac{1}{2} \|x_1 - x_2\|^2 + (1 - y) \cdot \frac{1}{2} \max(0,\ \alpha - \|x_1 - x_2\|)^2
$$


**Task**. Derive difference between **Triplet Loss**
$$
L_{triplet}(A, P, N)
$$

and two **Contrastive Losses** on specific pairs
$$
L_{contrastive}(A, P, 1) + L_{contrastive}(P, N, 0).
$$

How does it interpreted?




# Practice (20 points + 6 bonus point)

In the practical part of this homework we train ECAPA-TDNN model with some metric-based modifications:
1. Data discovery **(1 point)**
2. Implement ECAPA-TDNN **(4 points)**
3. Train ECAPA-TDNN **(4 points)**
4. Visualization: Clustering wav-speaker embeddings on test-set **(2 point)**
5. Implement Triplet Loss and Finetune ECAPA-TDNN model **(4 points)**
6. Implement Contrastive Loss and Finetune ECAPA-TDNN model **(3 points)**
7. Visualization: Clustering wav-speaker embeddings on test-set for Triplet loss and Contrastive Loss **(1 points)**
8. Answers **(1 point)**
9. VoxCeleb **(6 bonus point)**

In [ ]:
import os
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchaudio
from torchaudio import transforms as T
from IPython.display import clear_output
import matplotlib.pyplot as plt

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Discover your data (1 points)


Download LibriSpeech 100hr training and test data

In [ ]:
dataset_dir = './data'
if not os.path.isdir(dataset_dir):
    os.makedirs(dataset_dir)

train_dataset = torchaudio.datasets.LIBRISPEECH(dataset_dir, url="train-clean-100", download=True)
test_dataset = torchaudio.datasets.LIBRISPEECH(dataset_dir, url="test-clean", download=True) # it's not good practice, but for validation we will use the test split

After downloading the datasets, please, ask some questions:

1. How many audios and speakers do these datasets contain?

2. What is the correct way to split the data into train/test? (on your data) *Think about possibility of dataleaks in speakers*.

3. Is there any intersection between the speakers' set in the train and the test splits?

In [ ]:
### YOUR CODE HERE

....

### YOUR CODE HERE

Initialize the next global variable

In [ ]:
N_TRAIN_SPEAKER = # YOUR CODE HERE

# Dataset

In [ ]:
sample_rate = 16000
N_MELS = # YOUR CODE HERE

train_audio_transforms = nn.Sequential(
    T.MelSpectrogram(n_mels=N_MELS, sample_rate=sample_rate),
    # YOUR CODE HERE
)

test_audio_transforms = nn.Sequential(
    T.MelSpectrogram(n_mels=N_MELS, sample_rate=sample_rate)
)

**Question:** Which augmentation would you use in this homework? Which of them is more applicable?

Collator

In [ ]:
def get_path(speaker_id, dir, idx):
    return f'{speaker_id}_{dir}_{idx}'

In [ ]:
class Collate:
    def __init__(self, data_type = 'test') -> None:
        super(Collate, self).__init__()

        self.data_type = data_type

    def __call__(self, data: torchaudio.datasets.librispeech.LIBRISPEECH):
        spectrograms = []
        labels = []
        input_lengths = []
        paths = []
        for (waveform, _, _, speaker_id, dir, idx) in data:
            if self.data_type == 'train':
                spec = train_audio_transforms(waveform).squeeze(0).transpose(0, 1)
            elif self.data_type == 'test':
                spec = test_audio_transforms(waveform).squeeze(0).transpose(0, 1)
            else:
                raise Exception('data_type should be train or valid')

            spectrograms.append(torch.log(spec + 1e-3))
            label = torch.LongTensor([speaker_to_index[str(speaker_id)]])
            labels.append(label)
            input_lengths.append(spec.shape[0])
            paths.append(get_path(speaker_id, dir, idx))

        spectrograms = nn.utils.rnn.pad_sequence(spectrograms, batch_first=True).transpose(1, 2)
        labels = torch.cat(labels).view(-1, 1)
        return spectrograms, labels, input_lengths, paths

train_collate_fn = Collate(data_type='train')
test_collate_fn = Collate(data_type='test')

Dataloaders

In [ ]:
import numpy as np
import torch.utils.data as data
from tqdm import tqdm

In [ ]:
BATCH_SIZE = # YOUR CODE HERE


train_loader = data.DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    pin_memory=True,
    collate_fn=train_collate_fn,
    num_workers=4,
)

val_loader = data.DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=True,
    collate_fn=test_collate_fn,
    num_workers=4,
)

# Metric

In [ ]:
def cosine_similarity(a, b):
    a = a.reshape(-1)
    b = b.reshape(-1)
    return np.dot(a, b) / np.linalg.norm(a) / np.linalg.norm(b)


def best_eer(data):
    # data: [(cosine_distance, target), (cosine_distance, target), ...]
    full = sorted(data, key=lambda x: (x[0], -x[1]))
    pos = len([item for item in full if item[1] == 1])
    neg = len(full) - pos
    cur_pos = pos
    cur_neg = 0
    best_eer = 1
    for _, label in full:
        if label == 1:
            cur_pos -= 1
        else:
            cur_neg += 1
        cur_eer = max((pos - cur_pos) / pos, (neg - cur_neg) / neg)
        best_eer = min(best_eer, cur_eer)
    return best_eer

In [ ]:
# it will be used on eval stage
test_idx_to_speaker_id = {}
for (_, _, _, speaker_id, dir, idx) in tqdm(test_dataset):
    path = get_path(speaker_id, dir, idx)
    test_idx_to_speaker_id[path] = speaker_id

**Question:** What is EER metric? What does it mean?

# Implement ECAPA-TDNN (4 points)

Before coding, please, read [original paper](https://arxiv.org/pdf/2005.07143.pdf)

In [ ]:
class SEBlock(nn.Module):
    def __init__(self, input_shape: int, reduction: int):
        super().__init__()
        # YOUR CODE HERE
        ...

    def forward(self, X):
        # YOUR CODE HERE
        ...


class Res2Net(nn.Module):
    def __init__(self, hidden_dim: int, dilation: int, scale: int):
        super().__init__()
        # YOUR CODE HERE
        ...


    def forward(self, X):
        # YOUR CODE HERE
        ...

class EcapaBlock(nn.Module):
    def __init__(self, hidden_dim: int, dilation: int, scale: int):
        super().__init__()
        # YOUR CODE HERE
        ...

    def forward(self, X):
        # YOUR CODE HERE
        ...

class AttentiveStatsPooling(nn.Module):
    def __init__(self, input_shape: int, hidden_dim: int):
        super().__init__()
        # YOUR CODE HERE

    def forward(self, X):
        # X shape = [time, feats]
        # calc mean and std for X over time dimension
        # concatenate mean and std to X over feats dimension to make shape [time, feats * 3]
        # attention
        # weighted mean and std with weights from attention for original X

        # YOUR CODE HERE
        ...


class AAMSoftmax(nn.Module):
    def __init__(self, input_shape, n_class, margin, scale):
        super().__init__()
        # YOUR CODE HERE
        ...

    def forward(self, X):
        # calc cosine similarity between X and weights
        # YOUR CODE HERE
        ...


class EcapaTDNN(nn.Module):
    def __init__(self, input_shape: int, output_shape: int, hidden_dim: int):
        super().__init__()
        self.conv1 = nn.Conv1d(input_shape, 1024, kernel_size=5, stride=1, padding=2)
        self.bn1 = nn.BatchNorm1d(1024)

        self.block1 = EcapaBlock(1024, dilation=2, scale=8)
        self.block2 = EcapaBlock(1024, dilation=3, scale=8)
        self.block3 = EcapaBlock(1024, dilation=4, scale=8)

        self.conv_cat = nn.Conv1d(1024*3, 1024*3, kernel_size=1)
        self.bn_cat = nn.BatchNorm1d(1024*3)
        self.asp = AttentiveStatsPooling(1024*3, hidden_dim=hidden_dim)

        self.fc1 = nn.Linear(1024*3*2, 192)
        self.bn_fc1 = nn.BatchNorm1d(192)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = AAMSoftmax(192, output_shape, margin=0.2, scale=30)

    def forward(self, X, lengths):
        # YOUR CODE HERE
        ...

In [ ]:
model = EcapaTDNN(16, 64, 128)
input = torch.randn(4, 16, 1340)
out = model(input, None)

assert out[0].shape == (4, 64)
assert out[1].shape == (4, 128)

# Train ECAPA-TDNN model (4 points)

In [ ]:
def train_stage(model, loader, opt, scaler):
    loss_sum, batches = 0.0, 0
    for X, Y, lengths, _ in tqdm(loader):
        X, Y = X.to(DEVICE), Y.squeeze().to(DEVICE)

        with torch.amp.autocast(device_type='cuda'):
            logits, embds = # YOUR CODE HERE
            loss = # YOUR CODE HERE

        opt.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()

        loss_sum += loss.item()
        batches += 1

    return loss_sum / batches

In [ ]:
def eval_stage(model, loader, test_idx_to_speaker_id=test_idx_to_speaker_id):
    embeds = {}
    target_scores = []
    with torch.no_grad():
        for X, Y, lengths, paths in tqdm(loader):
            X, Y = X.to(DEVICE), Y.squeeze().to(DEVICE)
            with torch.amp.autocast(device_type='cuda'):
                _, embds = # YOUR CODE HERE

            embds = embds.cpu().data.numpy().reshape(X.shape[0], -1)
            for embd, path in zip(embds, paths):
                embeds[path] = embd

    for i, (p1, sp1) in tqdm(enumerate(test_idx_to_speaker_id.items())):
        for j, (p2, sp2) in enumerate(test_idx_to_speaker_id.items()):
            if j <= i:
                continue
            assert p1 in embeds and p2 in embeds

            target = 1 if sp1 == sp2 else 0
            target_scores.append((cosine_similarity(embeds[p1], embeds[p2]), target))
    return {
        'best_err': best_eer(target_scores)
    }

**Question:** Why don't we calculate the loss during validation?

In [ ]:
def train_loop(
    model: nn.Module,
    *,
    train_loader,
    val_loader,
    opt,
    scaler,
    epochs: int=10
):
    train_losses = []
    val_scores = []

    for epoch in range(epochs):
        model.train()
        train_losses.append(train_stage(model, loader=train_loader, opt=opt, scaler=scaler))

        model.eval()
        eval_results = eval_stage(model, loader=val_loader)
        val_scores.append(eval_results['best_err'])

        clear_output()

        fig, axis = plt.subplots(1, 2, figsize=(15, 7))
        axis[0].plot(np.arange(1, epoch + 2), train_losses, label='CE train loss', color='blue')
        axis[1].plot(np.arange(1, epoch + 2), val_scores, label='val score', color='red')
        axis[0].set(xlabel='epoch', ylabel='CE Loss')
        axis[1].set(xlabel='epoch', ylabel='EER')
        fig.legend()
        plt.show()
        print(f'Epoch {epoch + 1}. Train loss {train_losses[-1]}. Eval score {val_scores[-1]}')


        # YOUR CODE HERE
        # ... save best checkpoint
        # YOUR CODE HERE

In [ ]:
EPOCHS = # YOUR CODE HERE (at lease 5 epochs)
WEIGHT_DECAY = # YOUR CODE HERE
LR = # YOUR CODE HERE
HIDDENS = # YOUR CODE HERE


model = EcapaTDNN(N_MELS, N_TRAIN_SPEAKER, HIDDENS)
model = torch.nn.DataParallel(model).to(DEVICE)

scaler = torch.amp.GradScaler()
opt = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

train_loop(model, train_loader=train_loader, val_loader=val_loader, opt=opt, scaler=scaler)

# Visualization: Clustering wav-speaker embeddings on test-set (1 points)

In [ ]:
# YOUR CODE

...

# YOUR CODE

Question: Is it possible to associate clusters with a single speaker? Give an example

# Implement Triplet Loss and Finetune ECAPA-TDNN model (4 points)

*at least 2 epochs*

Do not forget about at least one positive sample in batch

In [ ]:
from collections import defaultdict

class PositivePairsSampler(torch.utils.data.Sampler):
    def __init__(self, speakers: list[int], batch_size: int):
        # YOUR CODE HERE
        ...

    def __len__(self):
        # YOUR CODE HERE
        ...

    def __iter__(self):
        # YOUR CODE HERE
        ...

# usage:
# loader = torch_data.DataLoader(
#        train_dataset,
#        collate_fn=dataset.collate_fn,
#        batch_sampler=PositivePairsSampler(train_dataset._speakers, batch_size),
#        ....kwargs
#    )

In [ ]:
# YOUR CODE

...

# YOUR CODE

# Implement Contrastive Loss and Finetune ECAPA-TDNN model (3 points)

*at least 2 epochs*

In [ ]:
# YOUR CODE

...

# YOUR CODE

# Visualization: Clustering wav-speaker embeddings on test-set for Triplet loss and Contrastive Loss (1 point)

Compare the results

In [ ]:
# YOUR CODE

...

# YOUR CODE

# Answers (1 point)

1. How do you like this HW? How much time (approximatelly) do you spend on this HW?
2. What was the most difficult part of these HW?
3. Where do you think models similar to ECAPA-TDNN are used in practice?
4. What other types of contrastive losses could you give? How do they differ from each other (briefly).

< YOUR ANSWERS HERE >

# VoxCeleb (6 bonus points)

1. Train ECAPA-TDNN on [VoxCeleb1 Dataset](https://docs.pytorch.org/audio/main/generated/torchaudio.datasets.VoxCeleb1Verification.html#torchaudio.datasets.VoxCeleb1Verification) (5 points)
    - Use 'dev' and 'test' subsplits
    - Modify the dataset/train/validation/metric code
    - Show train/validation learning curves
2. wav-speaker embeddings on test-set (1 point)

In [ ]:
# YOUR CODE

...

# YOUR CODE